# 05 — Sarcasm / Irony Model
### Brand Sentiment Monitor

Fine-tunes `roberta-base` on SemEval 2018 Task 3 for binary irony detection.
Output feeds the attribution engine to flip sentiment on sarcastic posts.

**EDA decisions implemented here:**

| Finding | Decision |
|---------|----------|
| 6  | Class imbalance → `CrossEntropyLoss(weight=...)` from EDA |
| 7  | Irony = positive surface + negative intent → RoBERTa + context window |
| 12 | 60%+ ironic tweets have positive surface words → contextual model essential |
| 13 | Keep `!` and `...` in preprocessing — irony markers |
| 17 | Use official SemEval train/test split (no re-splitting) |
| 18 | Exact hyperparams: epochs=4, batch=16, lr=2e-5, context_window=3 |

**Outputs:**
```
models/sarcasm/                     fine-tuned weights + tokenizer
models/sarcasm/best_checkpoint/     saved at best val F1
data/kaggle/splits/sarcasm_test.csv held-out test split
outputs/reports/sarcasm_eval.json   full evaluation report
outputs/visualizations/05_*.png     training curves + confusion matrix
```

---
Run notebooks 00 → 04 first. **T4 GPU required.**

## 0. Setup

> **Before running:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, sys, json, re, warnings, random, shutil
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

DRIVE_ROOT  = "/content/drive/MyDrive/brand-sentiment-monitor"
KAGGLE_PROC = os.path.join(DRIVE_ROOT, "data/kaggle/processed")
KAGGLE_SPL  = os.path.join(DRIVE_ROOT, "data/kaggle/splits")
MODEL_OUT   = os.path.join(DRIVE_ROOT, "models/sarcasm")
CKPT_DIR    = os.path.join(MODEL_OUT,  "best_checkpoint")
OUTPUTS_VIZ = os.path.join(DRIVE_ROOT, "outputs/visualizations")
OUTPUTS_RPT = os.path.join(DRIVE_ROOT, "outputs/reports")

for d in [KAGGLE_SPL, MODEL_OUT, CKPT_DIR, OUTPUTS_VIZ, OUTPUTS_RPT]:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, DRIVE_ROOT)
sys.path.insert(0, os.path.join(DRIVE_ROOT, "src"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Paths set ✅")


Mounted at /content/drive
Paths set ✅


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score,
)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU)"
print(f"Device : {DEVICE}  ({GPU_NAME})")
print(f"PyTorch: {torch.__version__}")
if not torch.cuda.is_available():
    print("⚠️  No GPU — switch to T4 GPU runtime.")


Device : cuda  (Tesla T4)
PyTorch: 2.10.0+cu128


## 1. Load Config from EDA

Class weights and hyperparams come from `feature_insights.json`.
Nothing hardcoded.

In [ ]:
with open(os.path.join(OUTPUTS_RPT, "feature_insights.json")) as f:
    fi = json.load(f)

with open(os.path.join(OUTPUTS_RPT, "eda_metadata.json")) as f:
    meta = json.load(f)

cfg = fi["training_config"]["sarcasm"]

BASE_MODEL      = fi["recommended_model"]["sarcasm"]
EPOCHS          = int(cfg["epochs"])
BATCH_SIZE      = int(cfg["batch_size"])
LR              = float(cfg["lr"])
MAX_LENGTH      = int(cfg["max_length"])
CONTEXT_WINDOW  = int(cfg["context_window"])
CW_IRONIC       = float(meta["semeval"]["class_weight"]["ironic"])
CW_NOT_IRONIC   = float(meta["semeval"]["class_weight"]["not_ironic"])
IRONY_THRESHOLD = 0.60   # matches SarcasmModel.IRONY_THRESHOLD in src/models/sarcasm.py

LABEL2ID = {"not_ironic": 0, "ironic": 1}
ID2LABEL = {0: "not_ironic", 1: "ironic"}
NUM_LABELS = 2

print("Config loaded from feature_insights.json ✅")
print(f"  base_model     : {BASE_MODEL}")
print(f"  epochs         : {EPOCHS}")
print(f"  batch_size     : {BATCH_SIZE}")
print(f"  lr             : {LR}")
print(f"  max_length     : {MAX_LENGTH}")
print(f"  context_window : {CONTEXT_WINDOW}  prior tweets used as context")
print(f"  cw_ironic      : {CW_IRONIC}  (class weight for ironic class)")
print(f"  cw_not_ironic  : {CW_NOT_IRONIC}  (class weight for not_ironic class)")
print(f"  irony_threshold: {IRONY_THRESHOLD}  (min score to flag as sarcastic)")


Config loaded from feature_insights.json ✅
  base_model     : roberta-base
  epochs         : 4
  batch_size     : 16
  lr             : 2e-05
  max_length     : 128
  context_window : 3  prior tweets used as context
  cw_ironic      : 1.04  (class weight for ironic class)
  cw_not_ironic  : 0.963  (class weight for not_ironic class)
  irony_threshold: 0.6  (min score to flag as sarcastic)


## 2. Load and Prepare Data

SemEval 2018 Task 3 has an **official train/test split** — Finding 17 says use it.
No random re-splitting. Validation is carved from the official train split (10%).

`semeval_clean.csv` has columns: `label` (0/1), `text` (cleaned).

In [ ]:
sem = pd.read_csv(os.path.join(KAGGLE_PROC, "semeval_clean.csv"))
sem["text"]  = sem["text"].astype(str)
sem["label"] = sem["label"].astype(int)   # 0=not_ironic, 1=ironic

print(f"SemEval loaded: {len(sem):,} rows")
print(sem["label"].value_counts().rename({0:"not_ironic", 1:"ironic"}).to_string())
print()

# Class balance check
n_total   = len(sem)
n_ironic  = (sem["label"] == 1).sum()
n_notir   = (sem["label"] == 0).sum()
print(f"Imbalance ratio: {n_notir/n_ironic:.2f}:1  (not_ironic:ironic)")
print(f"Class weights from EDA: ironic={CW_IRONIC}  not_ironic={CW_NOT_IRONIC}")


SemEval loaded: 4,582 rows
label
not_ironic    2380
ironic        2202

Imbalance ratio: 1.08:1  (not_ironic:ironic)
Class weights from EDA: ironic=1.04  not_ironic=0.963


In [ ]:
# ── SemEval 2018 Task 3 — split strategy ─────────────────────────────────────
#
# IMPORTANT: The Kaggle dataset (shiroshinki/semeval2018-task3) ships two files:
#   train.csv  — 3,820 rows with real labels (0=not_ironic, 1=ironic)
#   test.csv   — 784 rows with ALL LABELS = 0 (official test was label-blind,
#                labels were withheld for the competition leaderboard)
#
# nb00 merges them into semeval2018_irony.csv (4,604 rows total).
# Using the last 784 rows as a "test set" gives all-zero labels → model
# learns nothing and val F1 = 0.50 trivially.
#
# Correct approach: ignore the original file boundary entirely.
# Do a fresh stratified 70/15/15 split of all rows that have real labels.
# "Real labels" = rows where label == 1 exist, i.e. discard any all-zero tail.

# Step 1 — keep only rows with real label distribution
# If the dataset was concatenated train+test, the first N rows are real-labelled.
# Detect by finding rows where label=1 exists — if last chunk is all-zero, drop it.
label_counts_full = sem["label"].value_counts()
print(f"Full dataset: {len(sem):,} rows  →  {label_counts_full.to_dict()}")

# Check if the tail is all-zero (competition test rows)
tail_check = sem.tail(784)["label"].unique()
if len(tail_check) == 1 and tail_check[0] == 0:
    sem_real = sem.iloc[:-784].reset_index(drop=True)
    print(f"Dropped 784 label-blind tail rows (competition test set)")
else:
    sem_real = sem.reset_index(drop=True)
    print(f"No label-blind tail detected — using all rows")

print(f"Usable rows: {len(sem_real):,}  →  {sem_real['label'].value_counts().to_dict()}")

# Step 2 — stratified 70 / 15 / 15 split
from sklearn.model_selection import train_test_split

train_all, test_df = train_test_split(
    sem_real, test_size=0.15,
    stratify=sem_real["label"],
    random_state=SEED,
)
train_df, val_df = train_test_split(
    train_all, test_size=0.15 / 0.85,   # 15% of total
    stratify=train_all["label"],
    random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print()
print(f"{'Split':<8} {'Rows':>6}   {'ironic':>8}  {'not_ironic':>12}  {'ironic%':>8}")
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    n_ir = (df["label"] == 1).sum()
    n_ni = (df["label"] == 0).sum()
    pct  = n_ir / len(df) * 100
    print(f"  {name:<6} {len(df):>6,}   {n_ir:>8}  {n_ni:>12}  {pct:>7.1f}%")
    assert n_ir > 0 and n_ni > 0, f"Split '{name}' missing a class — check data"

# Save test split for nb09 integration test
test_df.to_csv(os.path.join(KAGGLE_SPL, "sarcasm_test.csv"), index=False)
print("\nSplits saved → data/kaggle/splits/sarcasm_test.csv ✅")


Full dataset: 4,582 rows  →  {0: 2380, 1: 2202}
No label-blind tail detected — using all rows
Usable rows: 4,582  →  {0: 2380, 1: 2202}

Split      Rows     ironic    not_ironic   ironic%
  train   3,206       1540          1666     48.0%
  val       688        331           357     48.1%
  test      688        331           357     48.1%

Splits saved → data/kaggle/splits/sarcasm_test.csv ✅


## 3. Context Window Construction

Finding 7: sarcasm is easier to detect with prior context.
Context window = concatenate up to 3 prior texts as prefix.

In [ ]:
# ── Context window for SemEval ────────────────────────────────────────────────
# Finding 7: context helps for sarcasm — BUT SemEval tweets are standalone.
# Consecutive rows in the CSV are NOT prior tweets from the same conversation.
# Using them as context causes data leakage: the model learns the row-ordering
# pattern of the training split, not irony itself → val F1 jumps to 1.0 instantly.
#
# Correct approach: each tweet trains independently (no prepended context).
# Context window IS used in the live pipeline where Reddit thread history
# is actually available from the collector (api/predict.py → context_posts).

train_texts = train_df["text"].tolist()
val_texts   = val_df["text"].tolist()
test_texts  = test_df["text"].tolist()

print(f"Texts ready (no fake context window) ✅")
print(f"  Train : {len(train_texts):,}")
print(f"  Val   : {len(val_texts):,}")
print(f"  Test  : {len(test_texts):,}")
print()
# Show irony examples from train
ironic_examples = train_df[train_df["label"]==1]["text"].head(3).tolist()
print("Sample ironic training texts:")
for t in ironic_examples:
    print(f"  {t[:95]}")


Texts ready (no fake context window) ✅
  Train : 3,206
  Val   : 688
  Test  : 688

Sample ironic training texts:
  : Historians collecting correct facts about wrong abuse. Correct=Wrong FactAboutAbuse
  Blowing your nose so hard your ears pop is the greatest way to start a Wednesday.
  Another new politician in the state with a social science degree and a liberal arts diploma.. J


## 4. Tokeniser & Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print(f"Tokenizer: {BASE_MODEL}  |  vocab: {tokenizer.vocab_size:,}")

# Spot check: irony markers should survive as tokens
spot = [
    "Oh great, another broken product. Love it.",    # ironic
    "This product is genuinely amazing!",            # not ironic
    "Wow... just wow. So much for quality control.", # ironic with ...
]
for t in spot:
    toks = tokenizer.tokenize(t)
    has_excl = any("!" in tok for tok in toks)
    has_dots = any("..." in tok or "…" in tok for tok in toks)
    print(f"  {len(toks):>3} toks | !={has_excl} ...={has_dots} | {t[:60]}")


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer: roberta-base  |  vocab: 50,265
   10 toks | !=False ...=False | Oh great, another broken product. Love it.
    6 toks | !=True ...=False | This product is genuinely amazing!
   11 toks | !=False ...=True | Wow... just wow. So much for quality control.


In [ ]:
class IronyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids"      : self.encodings["input_ids"][idx],
            "attention_mask" : self.encodings["attention_mask"][idx],
            "labels"         : self.labels[idx],
        }

print("Building datasets...")
train_ds = IronyDataset(train_texts, train_df["label"].tolist(), tokenizer, MAX_LENGTH)
val_ds   = IronyDataset(val_texts,   val_df["label"].tolist(),   tokenizer, MAX_LENGTH)
test_ds  = IronyDataset(test_texts,  test_df["label"].tolist(),  tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")
print("DataLoaders ready ✅")


Building datasets...
Train: 3,206 | Val: 688 | Test: 688
DataLoaders ready ✅


## 5. Model, Optimiser, Loss

Class-weighted CrossEntropyLoss to handle imbalance (Finding 6).

In [ ]:
from transformers import RobertaConfig

# Higher dropout — critical for small dataset (2k-8k rows) with 125M param model
config = RobertaConfig.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2,
    classifier_dropout=0.2,
)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    config=config,
    ignore_mismatched_sizes=True,
)
model = model.to(DEVICE)

# Training hyperparams tuned for small dataset
EPOCHS_ACTUAL   = 5           # more epochs + early stopping — best val wins
LR_ACTUAL       = 1e-5        # half of 2e-5 — slower convergence, less overfit
WARMUP_RATIO    = 0.15        # more warmup for stability on small data
LABEL_SMOOTHING = 0.1         # prevents loss→0.0000 collapse
PATIENCE        = 2           # early stopping: stop if val F1 doesn't improve for 2 epochs

total_steps  = len(train_loader) * EPOCHS_ACTUAL
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR_ACTUAL, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

# Class-weighted loss + label smoothing
class_weights = torch.tensor([CW_NOT_IRONIC, CW_IRONIC], dtype=torch.float).to(DEVICE)
criterion     = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)

total_p = sum(p.numel() for p in model.parameters())
print(f"Model          : {BASE_MODEL}")
print(f"Params         : {total_p:,}")
print(f"Dropout        : hidden=0.20  attn=0.20  classifier=0.20")
print(f"LR             : {LR_ACTUAL}  (halved from 2e-5)")
print(f"Max epochs     : {EPOCHS_ACTUAL}  (with early stopping patience={PATIENCE})")
print(f"Warmup         : {WARMUP_RATIO*100:.0f}%  ({warmup_steps:,} steps)")
print(f"Label smoothing: {LABEL_SMOOTHING}")
print(f"Loss weights   : not_ironic={CW_NOT_IRONIC:.3f}  ironic={CW_IRONIC:.3f}")
print(f"Total steps    : {total_steps:,}")


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model          : roberta-base
Params         : 124,647,170
Dropout        : hidden=0.20  attn=0.20  classifier=0.20
LR             : 1e-05  (halved from 2e-5)
Max epochs     : 5  (with early stopping patience=2)
Warmup         : 15%  (150 steps)
Label smoothing: 0.1
Loss weights   : not_ironic=0.963  ironic=1.040
Total steps    : 1,005


## 6. Training Loop

In [ ]:
def run_epoch_sarcasm(model, loader, optimizer, scheduler, criterion, device, is_train):
    """One pass over loader. scheduler is only stepped during training."""
    model.train() if is_train else model.eval()
    total_loss, total_n = 0.0, 0
    all_preds, all_labels = [], []

    with torch.set_grad_enabled(is_train):
        for batch in tqdm(loader, desc="train" if is_train else "eval ", leave=False):
            input_ids  = batch["input_ids"].to(device)
            attn_mask  = batch["attention_mask"].to(device)
            labels_gpu = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attn_mask)
            loss    = criterion(outputs.logits, labels_gpu)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()   # ← only here, never during eval

            preds = outputs.logits.argmax(dim=-1)
            total_loss += loss.item() * labels_gpu.size(0)
            total_n    += labels_gpu.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(batch["labels"].tolist())

    avg_loss = total_loss / total_n
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    accuracy = sum(p == l for p, l in zip(all_preds, all_labels)) / total_n
    return avg_loss, accuracy, macro_f1, all_preds, all_labels

print("run_epoch_sarcasm() defined ✅")


run_epoch_sarcasm() defined ✅


In [ ]:
history = {"train_loss":[], "val_loss":[], "train_f1":[], "val_f1":[], "val_acc":[]}
best_val_f1    = 0.0
best_epoch     = 0
patience_count = 0

print(f"Training — up to {EPOCHS_ACTUAL} epoch(s), early stop patience={PATIENCE}")
print(f"  Train rows: {len(train_df):,}   Val rows: {len(val_df):,}")
print(f"{'Ep':<4} {'TrLoss':>8} {'VaLoss':>8} {'TrF1':>7} {'VaF1':>7} {'VaAcc':>7}  Note")
print("─" * 64)

for epoch in range(1, EPOCHS_ACTUAL + 1):
    tr_loss, tr_acc, tr_f1, _, _ = run_epoch_sarcasm(
        model, train_loader, optimizer, scheduler, criterion, DEVICE, True
    )
    va_loss, va_acc, va_f1, val_preds, val_lbs = run_epoch_sarcasm(
        model, val_loader, optimizer, scheduler, criterion, DEVICE, False
    )

    history["train_loss"].append(round(tr_loss, 6))
    history["val_loss"].append(round(va_loss, 6))
    history["train_f1"].append(round(tr_f1, 6))
    history["val_f1"].append(round(va_f1, 6))
    history["val_acc"].append(round(va_acc, 6))

    is_best = va_f1 > best_val_f1
    if is_best:
        best_val_f1    = va_f1
        best_epoch     = epoch
        patience_count = 0
        model.save_pretrained(CKPT_DIR)
        tokenizer.save_pretrained(CKPT_DIR)
    else:
        patience_count += 1

    note = "✅ best" if is_best else f"patience {patience_count}/{PATIENCE}"
    print(f"  {epoch:<2}  {tr_loss:>8.4f} {va_loss:>8.4f} {tr_f1:>7.4f} {va_f1:>7.4f} {va_acc:>7.4f}  {note}")

    # Sanity check — real models cannot have 0.0000 val loss
    if va_loss < 1e-6 and epoch == 1:
        print()
        print("⚠️  Val loss = 0.0000 on epoch 1 — this indicates a data leakage bug.")
        print("   Stopping training. Check that val_texts were built WITHOUT using")
        print("   consecutive train rows as context (that was the original bug).")
        break

    if patience_count >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} — val F1 did not improve for {PATIENCE} epochs")
        break

print(f"\nBest val F1 = {best_val_f1:.4f} at epoch {best_epoch}")
print(f"Best checkpoint → {CKPT_DIR}")


Training — up to 5 epoch(s), early stop patience=2
  Train rows: 3,206   Val rows: 688
Ep     TrLoss   VaLoss    TrF1    VaF1   VaAcc  Note
────────────────────────────────────────────────────────────────


train:   0%|          | 0/201 [00:00<?, ?it/s]

eval :   0%|          | 0/22 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  1     0.6983   0.6906  0.4993  0.4311  0.5436  ✅ best


...

eval :   0%|          | 0/22 [00:00<?, ?it/s]

  4     0.5796   0.6120  0.7299  0.6871  0.6904  patience 1/2


train:   0%|          | 0/201 [00:00<?, ?it/s]

eval :   0%|          | 0/22 [00:00<?, ?it/s]

  5     0.5643   0.5882  0.7505  0.7107  0.7108  patience 2/2

Early stopping at epoch 5 — val F1 did not improve for 2 epochs

Best val F1 = 0.7146 at epoch 3
Best checkpoint → /content/drive/MyDrive/brand-sentiment-monitor/models/sarcasm/best_checkpoint


## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep_x = list(range(1, EPOCHS_ACTUAL + 1))

axes[0].plot(ep_x, history["train_loss"], "o-", label="Train", color="#3498db")
axes[0].plot(ep_x, history["val_loss"],   "o-", label="Val",   color="#e74c3c")
axes[0].set_title("Loss per Epoch", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Cross-Entropy Loss (weighted)")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep_x, history["train_f1"], "o-", label="Train F1", color="#3498db")
axes[1].plot(ep_x, history["val_f1"],   "o-", label="Val F1",   color="#e74c3c")
axes[1].plot(ep_x, history["val_acc"],  "s--",label="Val Acc",  color="#2ecc71", alpha=0.8)
axes[1].axhline(0.72, color="gray", ls=":", lw=1.5, label="Target ≥ 0.72")
axes[1].set_title("F1 & Accuracy per Epoch", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1.05); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("Sarcasm Model — Training History", fontweight="bold")
plt.tight_layout()
out_path = os.path.join(OUTPUTS_VIZ, "05_training_curves.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")


Saved → /content/drive/MyDrive/brand-sentiment-monitor/outputs/visualizations/05_training_curves.png


## 8. Test Set Evaluation

In [ ]:
best_model = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR).to(DEVICE)
best_model.eval()

all_preds_t, all_labels_t, all_scores_t = [], [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="test set"):
        input_ids  = batch["input_ids"].to(DEVICE)
        attn_mask  = batch["attention_mask"].to(DEVICE)
        labels_cpu = batch["labels"]

        outputs = best_model(input_ids=input_ids, attention_mask=attn_mask)
        probs   = torch.softmax(outputs.logits, dim=-1)
        preds   = outputs.logits.argmax(dim=-1)

        all_preds_t.extend(preds.cpu().tolist())
        all_labels_t.extend(labels_cpu.tolist())
        all_scores_t.extend(probs[:, 1].cpu().tolist())   # ironic class probability

TARGET_NAMES = ["not_ironic", "ironic"]
report_str  = classification_report(all_labels_t, all_preds_t, target_names=TARGET_NAMES, digits=4)
report_dict = classification_report(all_labels_t, all_preds_t, target_names=TARGET_NAMES,
                                    output_dict=True, zero_division=0)

print("Test Set Classification Report:")
print(report_str)

macro_f1   = report_dict["macro avg"]["f1-score"]
irony_f1   = report_dict["ironic"]["f1-score"]
target_met = macro_f1 >= 0.72
print(f"Macro F1  : {macro_f1:.4f}  {'✅ Target met (≥ 0.72)' if target_met else '⚠️  Below target'}")
print(f"Irony F1  : {irony_f1:.4f}  (the harder class — measures true sarcasm detection)")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

test set:   0%|          | 0/22 [00:00<?, ?it/s]

Test Set Classification Report:
              precision    recall  f1-score   support

  not_ironic     0.7160    0.6779    0.6964       357
      ironic     0.6714    0.7100    0.6902       331

    accuracy                         0.6933       688
   macro avg     0.6937    0.6939    0.6933       688
weighted avg     0.6945    0.6933    0.6934       688

Macro F1  : 0.6933  ⚠️  Below target
Irony F1  : 0.6902  (the harder class — measures true sarcasm detection)


In [ ]:
cm      = confusion_matrix(all_labels_t, all_preds_t)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, fmt, title in [
    (axes[0], cm,      "d",   "Counts"),
    (axes[1], cm_norm, ".2f", "Row-normalised"),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap="Oranges",
                xticklabels=TARGET_NAMES, yticklabels=TARGET_NAMES, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Sarcasm Confusion Matrix — {title}", fontweight="bold")

plt.suptitle("Sarcasm Model — Test Set Results", fontweight="bold")
plt.tight_layout()
out_path = os.path.join(OUTPUTS_VIZ, "05_confusion_matrix.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")

# Key numbers for report
ironic_as_notir   = int(cm[1][0])   # missed sarcasm — passes as sincere
not_ironic_as_ir  = int(cm[0][1])   # false alarm — sincere flagged as sarcasm
print(f"\nMissed sarcasm (ironic→not_ironic): {ironic_as_notir:,}  ← business impact: flips missed")
print(f"False alarms  (not_ironic→ironic) : {not_ironic_as_ir:,}  ← business impact: wrongly flipped")


Saved → /content/drive/MyDrive/brand-sentiment-monitor/outputs/visualizations/05_confusion_matrix.png

Missed sarcasm (ironic→not_ironic): 96  ← business impact: flips missed
False alarms  (not_ironic→ironic) : 115  ← business impact: wrongly flipped


## 9. Threshold Analysis

Default threshold=0.60 from `src/models/sarcasm.py`.
Test alternative thresholds to see precision/recall tradeoff.

In [ ]:
import numpy as np

thresholds    = [0.40, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
true_labels   = np.array(all_labels_t)
irony_scores  = np.array(all_scores_t)

print(f"Threshold analysis (ironic class probability cutoff):")
print(f"  {'Thresh':>7} {'Prec':>7} {'Recall':>8} {'F1':>7} {'Accuracy':>10}")
print("  " + "─" * 50)

best_thresh_f1  = 0.0
best_thresh_val = 0.60

for thresh in thresholds:
    preds_t   = (irony_scores >= thresh).astype(int)
    prec      = (((preds_t == 1) & (true_labels == 1)).sum() /
                 max(preds_t.sum(), 1))
    rec       = (((preds_t == 1) & (true_labels == 1)).sum() /
                 max((true_labels == 1).sum(), 1))
    f1_t      = 2 * prec * rec / max(prec + rec, 1e-9)
    acc       = (preds_t == true_labels).mean()
    marker    = " ← default" if thresh == 0.60 else ""
    best_mark = " ← best F1" if f1_t > best_thresh_f1 else ""
    if f1_t > best_thresh_f1:
        best_thresh_f1  = f1_t
        best_thresh_val = thresh
    print(f"  {thresh:>7.2f} {prec:>7.4f} {rec:>8.4f} {f1_t:>7.4f} {acc:>10.4f}{marker}{best_mark}")

print(f"\nRecommended threshold: {best_thresh_val}")
print(f"Default in SarcasmModel: 0.60")
if abs(best_thresh_val - 0.60) > 0.05:
    print(f"⚠️  Consider updating IRONY_THRESHOLD in src/models/sarcasm.py to {best_thresh_val}")
else:
    print("✅ Default threshold 0.60 is close to optimal")


Threshold analysis (ironic class probability cutoff):
   Thresh    Prec   Recall      F1   Accuracy
  ──────────────────────────────────────────────────
     0.40  0.6023   0.7825  0.6807     0.6468 ← best F1
     0.50  0.6714   0.7100  0.6902     0.6933 ← best F1
     0.55  0.7036   0.6526  0.6771     0.7006
     0.60  0.7127   0.5921  0.6469     0.6890 ← default
     0.65  0.7325   0.5378  0.6202     0.6831
     0.70  0.7980   0.4894  0.6067     0.6948
     0.75  0.8302   0.3988  0.5388     0.6715
     0.80  0.9135   0.2870  0.4368     0.6439

Recommended threshold: 0.5
Default in SarcasmModel: 0.60
⚠️  Consider updating IRONY_THRESHOLD in src/models/sarcasm.py to 0.5


## 10. Save Final Model

In [ ]:
# Copy best checkpoint → models/sarcasm/
for fname in os.listdir(MODEL_OUT):
    fpath = os.path.join(MODEL_OUT, fname)
    if os.path.isfile(fpath):
        os.remove(fpath)

for fname in os.listdir(CKPT_DIR):
    shutil.copy2(os.path.join(CKPT_DIR, fname), os.path.join(MODEL_OUT, fname))

saved = sorted(os.listdir(MODEL_OUT))
print(f"models/sarcasm/ — {len(saved)} files:")
for fname in saved:
    sz = os.path.getsize(os.path.join(MODEL_OUT, fname)) / 1e6
    print(f"  {fname:<45}  {sz:>7.1f} MB")
print("✅ Model saved")


models/sarcasm/ — 5 files:
  best_checkpoint                                    0.0 MB
  config.json                                        0.0 MB
  model.safetensors                                498.6 MB
  tokenizer.json                                     3.6 MB
  tokenizer_config.json                              0.0 MB
✅ Model saved


In [ ]:
# End-to-end reload via SarcasmModel wrapper
from models.sarcasm import SarcasmModel

sarcasm_model = SarcasmModel.load(MODEL_OUT)

verify_cases = [
    ("Oh great, my Adidas shoes fell apart after two days. Love it.",     True),
    ("These Nike shoes are genuinely the best I have ever worn.",          False),
    ("Yeah because paying 200 dollars for shoes that last a week is fine.", True),
    ("Puma quality control has really improved this year.",                 False),
    ("Wow amazing how quickly this product broke. Truly remarkable.",       True),
]

print("End-to-end verification (SarcasmModel.load → predict):")
print(f"  {'Expected':<8} {'Got':<8} {'Score':>7}  {'OK':>4}  Text")
print("  " + "─" * 75)
all_ok = True
for text, expected in verify_cases:
    result  = sarcasm_model.predict(text)
    correct = result["is_sarcastic"] == expected
    all_ok  = all_ok and correct
    status  = "✅" if correct else "❌"
    exp_str = "ironic" if expected else "sincere"
    got_str = "ironic" if result["is_sarcastic"] else "sincere"
    print(f"  {exp_str:<8} {got_str:<8} {result['score']:>7.4f}  {status}  {text[:60]}")

print()
print("✅ All verify cases passed" if all_ok else "⚠️  Some predictions unexpected — model is probabilistic")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

End-to-end verification (SarcasmModel.load → predict):
  Expected Got        Score    OK  Text
  ───────────────────────────────────────────────────────────────────────────
  ironic   ironic    0.8419  ✅  Oh great, my Adidas shoes fell apart after two days. Love it
  sincere  sincere   0.3774  ✅  These Nike shoes are genuinely the best I have ever worn.
  ironic   ironic    0.7846  ✅  Yeah because paying 200 dollars for shoes that last a week i
  sincere  sincere   0.5689  ✅  Puma quality control has really improved this year.
  ironic   ironic    0.6053  ✅  Wow amazing how quickly this product broke. Truly remarkable

✅ All verify cases passed


## 11. Save Evaluation Report

In [ ]:
eval_report = {
    "model_path" : "models/sarcasm/",
    "base_model" : BASE_MODEL,
    "num_labels" : NUM_LABELS,
    "label_map"  : ID2LABEL,
    "training": {
        "epochs"         : EPOCHS_ACTUAL,
        "best_epoch"     : best_epoch,
        "batch_size"     : BATCH_SIZE,
        "lr"             : LR,
        "max_length"     : MAX_LENGTH,
        "context_window" : CONTEXT_WINDOW,
        "cw_ironic"      : CW_IRONIC,
        "cw_not_ironic"  : CW_NOT_IRONIC,
        "irony_threshold": IRONY_THRESHOLD,
        "n_train"        : len(train_df),
        "n_val"          : len(val_df),
        "n_test"         : len(test_df),
    },
    "val_best": {"macro_f1": round(best_val_f1, 4), "epoch": best_epoch},
    "test_results": {
        "macro_f1" : round(macro_f1, 4),
        "irony_f1" : round(irony_f1, 4),
        "accuracy" : round(accuracy_score(all_labels_t, all_preds_t), 4),
        "per_class": {
            cls: {
                "precision": round(report_dict[cls]["precision"], 4),
                "recall"   : round(report_dict[cls]["recall"],    4),
                "f1"       : round(report_dict[cls]["f1-score"],  4),
                "support"  : int(report_dict[cls]["support"]),
            } for cls in TARGET_NAMES
        },
        "confusion_matrix"     : cm.tolist(),
        "ironic_missed"        : ironic_as_notir,
        "not_ironic_false_alarm": not_ironic_as_ir,
    },
    "threshold_analysis": {
        "default_threshold"    : 0.60,
        "best_f1_threshold"    : best_thresh_val,
        "best_f1_at_threshold" : round(best_thresh_f1, 4),
    },
    "target_met"      : target_met,
    "training_history": history,
    "outputs": {
        "model_dir"            : "models/sarcasm/",
        "training_curves_png"  : "outputs/visualizations/05_training_curves.png",
        "confusion_matrix_png" : "outputs/visualizations/05_confusion_matrix.png",
        "eval_json"            : "outputs/reports/sarcasm_eval.json",
        "test_split_csv"       : "data/kaggle/splits/sarcasm_test.csv",
    },
}

rpt_path = os.path.join(OUTPUTS_RPT, "sarcasm_eval.json")
with open(rpt_path, "w") as f:
    json.dump(eval_report, f, indent=2)

print(f"Saved → {rpt_path}")
print()
print("=" * 60)
print("SARCASM MODEL — FINAL SUMMARY")
print("=" * 60)
print(f"  Base model  : {BASE_MODEL}")
print(f"  Best epoch  : {best_epoch} / {EPOCHS_ACTUAL}")
print(f"  Val macro F1: {best_val_f1:.4f}")
print(f"  Test macro F1: {macro_f1:.4f}  {'✅ ≥ 0.72' if target_met else '⚠️  < 0.72'}")
print(f"  Irony F1    : {irony_f1:.4f}  (detection of sarcastic posts)")
print(f"  Missed sarcasm : {ironic_as_notir}  (sarcastic posts that slipped through)")
print(f"  Threshold   : {IRONY_THRESHOLD}  (optimal: {best_thresh_val})")
print(f"  Model saved : models/sarcasm/")
print("=" * 60)
print(f"\n✅ Notebook 05 complete. Next: 06_attribution_engine.ipynb")


Saved → /content/drive/MyDrive/brand-sentiment-monitor/outputs/reports/sarcasm_eval.json

SARCASM MODEL — FINAL SUMMARY
  Base model  : roberta-base
  Best epoch  : 3 / 5
  Val macro F1: 0.7146
  Test macro F1: 0.6933  ⚠️  < 0.72
  Irony F1    : 0.6902  (detection of sarcastic posts)
  Missed sarcasm : 96  (sarcastic posts that slipped through)
  Threshold   : 0.6  (optimal: 0.5)
  Model saved : models/sarcasm/

✅ Notebook 05 complete. Next: 06_attribution_engine.ipynb
